# Optional method: sentiment arcs (the emotional shape of a story)

An optional starter for **story projects**.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset (mount Drive; falls back to a local folder) ---
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
%pip install -q vaderSentiment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    _an = SentimentIntensityAnalyzer()
    def sentiment(s): return _an.polarity_scores(s)["compound"]
    print("using VADER sentiment")
except Exception as e:
    # Tiny offline fallback lexicon so the notebook runs in the harness.
    POS=set("happy joy love bright hope warm win light calm safe gentle".split())
    NEG=set("sad fear dark death cold loss cry pain fight alone broken".split())
    def sentiment(s):
        w=s.lower().split(); 
        return (sum(t in POS for t in w)-sum(t in NEG for t in w))/(len(w) or 1)
    print("using offline fallback lexicon:", type(e).__name__)

In [ ]:
import os


## Your story, sliced into segments

In class this is your narrative.

In [ ]:
import os, re

def load_gutenberg(url):
    """Fetch from Project Gutenberg and strip the boilerplate."""
    import requests
    raw = requests.get(url, timeout=30).text.replace("\r\n", "\n")
    body = re.split(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG.*?\*\*\*", raw, flags=re.S)[-1]
    return re.split(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG", body)[0]

story = load_gutenberg("https://www.gutenberg.org/cache/epub/84/pg84.txt")
print(len(story.split()), "words of Frankenstein: a real emotional arc to trace")

# Split the novel into 100 equal segments and score each: the emotional arc.
words = story.split()
seg = max(1, len(words) // 100)
segments = [" ".join(words[i:i+seg]) for i in range(0, len(words), seg)][:100]
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    an = SentimentIntensityAnalyzer()
    raw = np.array([an.polarity_scores(s_)["compound"] for s_ in segments])
except Exception as e:
    print("vaderSentiment unavailable, scoring with a small word list:", e)
    POS = {"love","hope","joy","happy","light","dear","sweet","beautiful","friend","gentle"}
    NEG = {"death","fear","misery","wretch","horror","dark","despair","grief","pain","monster"}
    def score(t):
        toks = t.lower().split()
        return (sum(w in POS for w in toks) - sum(w in NEG for w in toks)) / max(1, len(toks)) * 50
    raw = np.array([score(s_) for s_ in segments])
print(len(raw), "segments scored")

## Smooth it - and watch the window change the shape

A rolling average reveals the arc by averaging out segment-to-segment noise.

*(In Colab you can make this a live `ipywidgets` slider; here we compare two fixed windows so the
notebook is reproducible.)*

In [ ]:
def smooth(x, w):
    if w<=1: return x
    k=np.ones(w)/w
    return np.convolve(x, k, mode="same")

plt.figure(figsize=(7,4))
plt.plot(raw, ".", color="#999", label="raw segments")
plt.plot(smooth(raw,3), label="window=3 (light)")
plt.plot(smooth(raw,7), label="window=7 (heavy)")
plt.axhline(0, color="#ccc", lw=0.8)
plt.title("Emotional arc - the smoothing window changes the shape")
plt.xlabel("story segment"); plt.ylabel("sentiment"); plt.legend(); plt.tight_layout(); plt.show()

## You made an emotional arc

Change `N_SEG` and the smoothing windows and watch the arc breathe.

**Sketch:** plot your own story's arc at two smoothing windows; say where the shape is real and
where the smoothing might be inventing it.